## Model API

The `VCO_Model` class encapsulates all relevant relationships:

- `fosc_from_vin(v_in)` – compute oscillation frequency for a given input voltage
- `vin_from_fosc(f_osc)` – invert the model to extract input voltage
- `conductance(v_in, i_dc)` – calculate conductance from voltage and current
- `dG_df(v_in, i_dc)` – dG/d(f_osc) at a given operating point; quantifies G uncertainty due to f_osc measurement error
- `dG_df_curve(i_dc_range)` – matrix of G tolerances (dG/d(f_osc)) across a range of currents and voltages

You can instantiate the model with your measured datasets and then call these methods directly for analysis or testing, independent of the plotting routines.

# PoI Plotter

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

## VCO_Model class

In [2]:
from VCO_model import VCO_Model
# instantiate the model with our data
VCO_model = VCO_Model(
    vin_data=[225,250,275,300,330,340,360,380,400,420,440,460,480,500,
              520,540,560,580,600,620,640,660,680,700,720,740,760,780,800,820],
    fosc_data=[0,0,0,0,24,26,31,37,45,55,67,81,99,120,146,174,209,247,
               292,343,397,455,520,599,676,760,855,950,1050,1170]
)
# Measured data from your graph
vin_data = VCO_model.vin_data
fosc_data = VCO_model.fosc_data


In [3]:
# from pannels import plot_fosc_model
# plot_fosc_model(VCO_model, mode="fit")

In [4]:
from pannels import *
from workflow import compute_workflow

def PoI_plotter():
    def plot_workflow(f_osc_measured_kHz, i_dc_uA, fs_Hz):
        # Build result from interactive inputs
        result = compute_workflow(VCO_model, f_osc_measured_kHz, i_dc_uA, fs_Hz)

        fig = plt.figure(figsize=(16, 12))
        gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)

        plot_fosc_model(VCO_model, mode = "measurement", ax=fig.add_subplot(gs[0, 0]), result=result)
        plot_vin_text(fig.add_subplot(gs[0, 1]), VCO_model, result)
        plot_G_vs_idc(fig.add_subplot(gs[1, 0]), result)
        plot_conductance_text(fig.add_subplot(gs[1, 1]), result)
        plot_dGdf(fig.add_subplot(gs[2, 0]), result)
        plot_delta_G(fig.add_subplot(gs[2, 1]), result, fs_Hz)

        plt.suptitle('Integrated Measurement Workflow: f_osc → V_in → G',fontsize=14, fontweight='bold', y=0.995)
        plt.show()

        print_analysis(VCO_model, result)

        # Create interactive controls
    f_osc_slider = FloatSlider(value=820,min=0,max=1200,step=10,description='f_osc (kHz):',continuous_update=True)
    i_dc_slider = FloatSlider(value=1.0,min=0.1,max=10,step=0.1,description='i_dc (μA):',continuous_update=True)
    fs_Hz_slider = FloatSlider(value=1,min=0.5,max=20,step=0.5,description='fs (Hz):',continuous_update=True)
    
    interact(plot_workflow, 
             f_osc_measured_kHz=f_osc_slider,
             i_dc_uA=i_dc_slider,
             fs_Hz=fs_Hz_slider)

In [ ]:
PoI_plotter()

interactive(children=(FloatSlider(value=820.0, description='f_osc (kHz):', max=1200.0, step=10.0), FloatSlider…